In [2]:
import json
import pandas as pd

file_path = '../SSU_Datathon2025_공학분야_62199.json'  # 실제 파일 경로로 수정하세요

try:
    # 1. json 라이브러리로 파일 읽기
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 2. 데이터 구조에 따라 DataFrame 생성
    if isinstance(data, list):
        # 가장 일반적인 경우: [{...}, {...}] 형태
        df = pd.DataFrame(data)
        
    elif isinstance(data, dict):
        # 딕셔너리인 경우: {"response": [...]} 또는 {"data": [...]} 형태일 가능성 높음
        # 값이 '리스트'인 키를 자동으로 찾습니다.
        target_list = None
        for key, value in data.items():
            if isinstance(value, list):
                target_list = value
                print(f"알림: '{key}' 키 안에 있는 데이터 리스트를 사용합니다.")
                break
        
        if target_list:
            df = pd.DataFrame(target_list)
        else:
            # 리스트를 못 찾았다면 단일 객체로 간주
            df = pd.DataFrame([data])
            
    else:
        raise ValueError("지원하지 않는 데이터 형식입니다.")

    # 3. 데이터 전처리 및 집계
    # 컬럼명 공백 제거 (안전을 위해)
    df.columns = df.columns.str.strip()

    if 'PBSH' in df.columns and 'NODE_CLSS_02' in df.columns:
        # 연도 추출 (PBSH 앞 4자리)
        df['Year'] = df['PBSH'].astype(str).str[:4]

        # 집계 (groupby)
        result = df.groupby(['Year', 'NODE_CLSS_02']).agg(
            논문수=('NODE_ID', 'count'),
            학회=('PLCT_NM', lambda x: ', '.join(sorted(x.unique())))
        ).reset_index()

        # 4. 결과 출력
        print("-" * 50)
        print(result)
        print("-" * 50)
        
        # (선택) CSV 파일로 저장
        result.to_csv('result.csv', index=False, encoding='utf-8-sig')
        
    else:
        print("오류: 'PBSH' 또는 'NODE_CLSS_02' 컬럼을 찾을 수 없습니다.")
        print(f"현재 로드된 컬럼 목록: {df.columns.tolist()}")

except json.JSONDecodeError:
    print("JSON 파일 형식이 올바르지 않습니다. (괄호 짝이 맞는지, 쉼표가 올바른지 확인해주세요)")
except FileNotFoundError:
    print(f"파일을 찾을 수 없습니다: {file_path}")
except Exception as e:
    print(f"예상치 못한 오류가 발생했습니다: {e}")

알림: 'NODE_LIST' 키 안에 있는 데이터 리스트를 사용합니다.
--------------------------------------------------
    Year NODE_CLSS_02   논문수                                                 학회
0   2021         건축공학  2402  ARCHITECTURAL RESEARCH, Environmental Engineer...
1   2021        공학 일반  1128                   센서학회지, 한국산업정보학회논문지, 한국산학기술학회 논문지
2   2021         기계공학  2635  International Journal of Aerospace System Engi...
3   2021        기타 공학   447  교통기술과정책, 대한교통학회지, 재활복지공학회논문지, 철도저널, 한국게임학회 논문지...
4   2021         산업공학   289  Industrial Engineering & Management Systems, I...
5   2021     재료·에너지공학   308  Current Photovoltaic Research, 소성·가공, 신·재생에너지,...
6   2021       전기전자공학  3697  IEIE Transactions on Smart Processing & Comput...
7   2021       조선해양공학   208       대한조선학회 논문집, 대한조선학회지, 한국해양공학회지, 한국해양환경·에너지학회지
8   2021         컴퓨터학  1327  JOURNAL OF PLATFORM TECHNOLOGY, Journal of Com...
9   2021         화학공학   333       BT NEWS, KSBB Journal, 고분자 과학과 기술, 청정기술, 폴리머
10  2022         건축공학  2333  ARCHITECTUR

In [3]:
import pandas as pd

# 1. 파일 경로 설정 (사용자 PC 환경에 맞게)
file_result = 'result.csv'
file_paths_if = {
    2021: '2021_인용지수_2년분.xls',
    2022: '2022_인용지수_2년분.xls',
    2023: '2023_인용지수_2년분.xls',
    2024: '2024_인용지수_2년분.xls'
}

# 2. 결과 파일(result.csv) 로드
try:
    df_result = pd.read_csv(file_result)
    df_result.columns = df_result.columns.str.strip()
    print("✅ result.csv 로드 완료")
except Exception as e:
    print(f"❌ result.csv 로드 실패: {e}")
    exit()

# 3. 인용지수 파일 로드 (xlrd 엔진 사용)
if_data = {}

def load_xls_file(year, path):
    try:
        # engine='xlrd' 필수 (.xls 파일 처리용)
        df = pd.read_excel(path, engine='xlrd')
        
        # 컬럼명 전처리
        df.columns = df.columns.str.replace('\n', '').str.strip()
        
        # IF 컬럼 찾기 ('2년'과 'IF'가 포함된 컬럼)
        if_col = [c for c in df.columns if '2년' in c and 'IF' in c][0]
        
        # 숫자 변환 (오류값은 0으로) 및 중복 제거
        df[if_col] = pd.to_numeric(df[if_col], errors='coerce').fillna(0)
        df = df.sort_values(by=if_col, ascending=False).drop_duplicates(subset=['학술지명'])
        
        return dict(zip(df['학술지명'], df[if_col]))
        
    except Exception as e:
        print(f"⚠️ {year}년 로드 실패: {e}")
        return {}

# 연도별 데이터 로드 실행
for year in [2021, 2022, 2023, 2024]:
    if_data[year] = load_xls_file(year, file_paths_if[year])

# 2025년은 2024년 데이터 사용
if_data[2025] = if_data.get(2024, {})

# 4. 순위 매기기 함수
def rank_journals(row):
    try:
        year = int(float(row['Year']))
    except:
        return ""
    
    journals_str = row['학회']
    if pd.isna(journals_str) or journals_str == "":
        return ""
    
    year_map = if_data.get(year, {})
    
    # 리스트 생성
    ranked_list = []
    for journal in [j.strip() for j in journals_str.split(',')]:
        score = year_map.get(journal, 0)
        ranked_list.append((journal, score))
    
    # 정렬: 점수 내림차순 -> 이름 오름차순
    ranked_list.sort(key=lambda x: (-x[1], x[0]))
    
    # 줄바꿈(\n)으로 연결하여 문자열 반환
    return "\n".join([f"{i+1}. {name} ({score})" for i, (name, score) in enumerate(ranked_list)])

# 순위 컬럼 생성
df_result['학회_순위(IF)'] = df_result.apply(rank_journals, axis=1)

# 5. 필요한 컬럼만 선택하여 저장 (핵심 수정 사항)
# '학회' 컬럼은 제외하고 저장
final_columns = ['Year', 'NODE_CLSS_02', '논문수', '학회_순위(IF)']

# 만약 '논문수' 컬럼 이름이 다르다면(예: count) 확인 후 수정 필요
# 현재 result.csv 헤더: Year, NODE_CLSS_02, 논문수, 학회
# 따라서 위 컬럼명 그대로 사용하면 됨

output_file = 'result_ranked_final.csv'
df_result[final_columns].to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"\n🎉 '{output_file}' 저장 완료!")

✅ result.csv 로드 완료
WARNING *** OLE2 stream 'SSCS': expected size 132288, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 133824, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 132288, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 133312, actual size 512

🎉 'result_ranked_final.csv' 저장 완료!
   (불필요한 '학회' 컬럼은 제외되었습니다.)
